# Valoração

Este notebook documenta o passo a passo do processo de valoração da ferramenta de Valoração. O objetivo é criar um pipeline limpo, rastreável e performático para ler, processar e calcular as regras de preços e impostos (N13).

## Importação das Bibliotecas

*   **`pandas`**: Biblioteca principal para manipulação e análise de dados. Será usada para ler as planilhas, realizar os cruzamentos de tabelas (merges) e aplicar as fórmulas de valoração.
*   **`Path` (da biblioteca `pathlib`)**: Facilita a gestão de caminhos de arquivos de forma robusta e multiplataforma (garante que o código funcione sem alterações no Windows, Linux ou macOS).
*   **`time`**: Biblioteca nativa do Python utilizada para medir o tempo de processamento de cada bloco, permitindo identificar gargalos de performance.
*   **`numpy`**: Biblioteca para computação científica. Será extremamente útil para aplicar lógicas condicionais rápidas e gerenciar valores ausentes ou nulos.

In [ ]:
import pandas as pd
from pathlib import Path
import time
import numpy as np

## Criação da Base N13P

Nesta etapa, realizamos a leitura otimizada das nossas três bases de dados de entrada: **Ciclo N13P**, **Clientes** e **Produtos**. 

Para maximizar a performance e a segurança do pipeline, aplicamos as seguintes práticas:
*   **Engine Calamine**: Leitura ultra-rápida de planilhas Excel utilizando uma engine baseada em Rust.
*   **Seleção de Colunas**: Importamos apenas as colunas estritamente necessárias para o cálculo de valoração, economizando memória.
*   **Tipagem Estrita**: Forçamos IDs e códigos identificadores como texto (`string`/`object`) para evitar que o Python remova zeros à esquerda.
*   **Precisão Decimal**: Todos os valores numéricos flutuantes (`float`) serão exibidos e tratados com precisão de 4 casas decimais.


In [ ]:
# global setting pandas
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if isinstance(x, (int, float)) else str(x))

period = 3
year = 2026


# path to the files
caminho_dados = Path('../data/')

arquivo_n13p = caminho_dados / f"Ciclo_P{period:02d} N13P {year} - envio.xlsx"
arquivo_clientes = caminho_dados / 'BASE CLIENTES.xlsx'
arquivo_produtos = caminho_dados / 'BASE PRODUTOS.xlsx'

# columns names and types 
colunas_n13p = {
    'Tipo 1': str,
    'Tipo 2': str,
    'Tipo 3': str,
    'Regional': str,
    'GP': str,
    'Vend.': str,
    'Gerente': str,
    'Rede': str,
    'COD_CLIENTE': str,
    'Company Code': str,
    'CD': str,
    'NOME_CLIENTE': str,
    'UF': str,
    'Região': str,
    'EAN': str,
    'SKU': str,
    'Desc. SKU': str,
    'Classificação': str,
    'Tech': str,
    'Tech 2': str,
    'Subbrand': str,
    'Size': str,
    'Nivel 3 HieraR': str,
    'Marca': str
}
colunas_periodos = [f'P{i:02d}-{year}' for i in range(period, 14)]

for col_periodo in colunas_periodos:
    colunas_n13p[col_periodo] = float


colunas_clientes = {
    'COD_CLIENTE': str,
    'NOME_CLIENTE': str,
    'UF': str,
    'Rede': str,
    'COD REDE': str,
    'COD SUBREDE': str,
    'COND. PAG': str,
    'GP': str,
    'COD GP': str
}

colunas_produtos = {
    'EAN': str,
    'Descrição': str,
    'SKU': str,
    'Family Price': str,
    'Categoria': str,
    'Brand': str,
    'Sub Brand': str,
    'Ton/CDA': float,
    'Unid/CX': float, # originalmente é 'Unid/\nCX	': float
    'Origem': str,
    'Hierarquia': str,
    'NCM': str,
    'Tipo': str,
    'Promoção': str,
    'Class.': str,
    'kg/Un': float,
    'H05': str,
    'LSV': float,
}

# read files
start_time = time.time()

df_n13p = pd.read_excel(arquivo_n13p, engine='calamine', header=3, usecols=colunas_n13p.keys(), dtype=colunas_n13p)
df_n13p.rename(columns={'UF': 'UF DESTINO'}, inplace=True)

df_clientes = pd.read_excel(arquivo_clientes, engine='calamine', header=0, usecols=colunas_clientes.keys(), dtype=colunas_clientes)
df_produtos = pd.read_excel(arquivo_produtos, engine='calamine', header=0, usecols=colunas_produtos.keys(), dtype=colunas_produtos)

# merges

# Célula 4
inicio_ciclo = time.time()

dicionario_uf = {
    'BR01': 'SP',
    'BR03': 'PE',
    'BR30': 'SP',
    'BR31': 'MG',
}

df_n13p['UF ORIGEM'] = df_n13p['CD'].map(dicionario_uf)


# ==============================================================================
# 1. GERAÇÃO DO EAN ESPELHO
# ==============================================================================
search_ean = df_produtos.set_index('EAN', drop=False)['EAN'].to_dict()
df_n13p['EAN Espelho'] = df_n13p['EAN'].map(search_ean)

search_desc = df_produtos.set_index('Descrição', drop=False)['EAN'].to_dict()
secondary_search = df_n13p['Desc. SKU'].map(search_desc)
df_n13p['EAN Espelho'] = df_n13p['EAN Espelho'].fillna(secondary_search)


# ==============================================================================
# 2. ENRIQUECIMENTO DE PRODUTOS (Merge Exato + Merge de Resgate)
# ==============================================================================
# Colunas que queremos extrair da base de produtos
colunas_produtos_selecionadas = [
    'EAN', 'SKU', 'Family Price', 'Hierarquia', 'Class.', 'NCM', 
    'Origem', 'kg/Un', 'Ton/CDA', 'Unid/CX', 'LSV'
]

# Base para cruzamento exato (EAN + SKU)
df_prod_exato = df_produtos[colunas_produtos_selecionadas].drop_duplicates(subset=['EAN', 'SKU'], keep='first')

# Base de resgate (apenas por EAN, removendo a coluna SKU)
df_prod_resgate = df_produtos[colunas_produtos_selecionadas].drop(columns=['SKU']).drop_duplicates(subset=['EAN'], keep='first')

# Primeiro Merge: Cruzamento Exato
df_n13p = pd.merge(
    df_n13p,
    df_prod_exato,
    left_on=['EAN Espelho', 'SKU'],
    right_on=['EAN', 'SKU'],
    how='left'
)

# Ajuste pós-merge das colunas EAN duplicadas
df_n13p = df_n13p.drop(columns=['EAN_y'], errors='ignore')
df_n13p = df_n13p.rename(columns={'EAN_x': 'EAN'})

# Segundo Merge: Resgate pelo EAN Espelho
df_n13p = pd.merge(
    df_n13p,
    df_prod_resgate,
    left_on='EAN Espelho',
    right_on='EAN',
    how='left',
    suffixes=('', '_resgate')
)

# Preenchimento de lacunas (fillna) usando os dados de resgate
colunas_preencher = ['Family Price', 'Hierarquia', 'Class.', 'NCM', 'Origem', 'kg/Un', 'Ton/CDA', 'Unid/CX', 'LSV']
for col in colunas_preencher:
    df_n13p[col] = df_n13p[col].fillna(df_n13p[f'{col}_resgate'])

# Eliminação das colunas temporárias de resgate
colunas_lixo = [f'{col}_resgate' for col in colunas_preencher] + ['EAN_resgate']
df_n13p = df_n13p.drop(columns=colunas_lixo, errors='ignore')


# ==============================================================================
# 3. ENRIQUECIMENTO DE CLIENTES (Subrede e Condição de Pagamento)
# ==============================================================================
df_clientes_limpo = df_clientes[['COD_CLIENTE', 'COD REDE', 'COD SUBREDE', 'COND. PAG', 'COD GP']].drop_duplicates(subset=['COD_CLIENTE'], keep='first')

df_n13p = pd.merge(
    df_n13p,
    df_clientes_limpo, 
    on='COD_CLIENTE',
    how='left'
)

fim_ciclo = time.time()
tempo_ciclo = fim_ciclo - inicio_ciclo

print(f"⏱️ Tempo de processamento do ciclo e merges: {tempo_ciclo:.4f} segundos")
print(f"✅ Base Ciclo N13P enriquecida com Produtos e Clientes!")
display(df_n13p[['COD_CLIENTE', 'COD REDE', 'COD SUBREDE', 'SKU', 'EAN Espelho', 'LSV', 'COND. PAG']].head())



end_time = time.time()

print(f"⏱️ Tempo de leitura e tratamento inicial: {end_time - start_time:.4f} segundos")


In [ ]:
df_n13p['EAN Espelho']

## Carregamento das Bases das ZPs

Nesta etapa, carregamos as bases de dados de **ZPs** utilizando o mesmo padrão otimizado com a engine **Calamine**. Estas tabelas contêm as regras de precificação, listas de preço e exceções comerciais necessárias para o motor de valoração.


In [ ]:
# paths
path_zp55 = caminho_dados / 'ZP55.xlsx'
path_zp54 = caminho_dados / 'ZP54.xlsx'
path_zp53 = caminho_dados / 'ZP53.xlsx'
path_zp52 = caminho_dados / 'ZP52.xlsx'
path_zp73 = caminho_dados / 'ZP73.xlsx'
path_zp70 = caminho_dados / 'ZP70.xlsx'
path_zp39 = caminho_dados / 'ZP39.xlsx'

# columns and types
colunas_zp55 = {
    'CHAVE': str,
    'Cadastro SAP': str,
    'Org. Vendas': str,
    'Emissor': str,
    'País': str,
    'Centro': str,
    'UF': str,
    'NCM': str,
    'H01': str,
    'H02': str,
    'H03': str,
    'H04': str,
    'H05': str,
    'H06': str,
    'Cadastro': float,
}

colunas_zp54 = {
    'CHAVE': str,
    'Cadastro SAP': str,
    'Org. Vendas': str,
    'Emissor': str,
    'Rede': str,
    'GP': str,
    'UF': str,
    'H01': str,
    'H02': str,
    'H03': str,
    'H04': str,
    'H05': str,
    'H06': str,
    'Cadastro': float,
}

colunas_zp53 = {
    'CHAVE': str,
    'Cadastro SAP': str,
    'Org. Vendas': str,
    'Emissor': str,
    'Rede': str,
    'GP': str,
    'UF': str,
    'H01': str,
    'H02': str,
    'H03': str,
    'H04': str,
    'H05': str,
    'H06': str,
    'Cadastro': float,
    'Validade': str,
    'P\'ANO_FIM': str,
}

colunas_zp52 = {
    'CHAVE': str,
    'Cadastro SAP': str,
    'Org. Vendas': str,
    'Rede': str,
    'H01': str,
    'Cadastro': float,
    'Validade': str,
}

colunas_zp73 = {
    'CHAVE': str,
    'Cadastro SAP': str,
    'Org. Vendas': str,
    'Emissor': str,
    'Rede': str,
    'H01': str,
    'Cadastro': float,
    'Validade': str,
}

colunas_zp70 = {
    'CONDICAO DE PAGAMENTO': str,
    'Desconto': float,
}

colunas_zp39 = {
    'CHAVE': str,
    'Cadastro SAP': str,
    'Org. Vendas': str,
    'Emissor': str,
    'H01': str,
    'H02': str,
    'H03': str,
    'H04': str,
    'H05': str,
    'Cadastro': float,
    'P\'ANO_FIM': str,
}

df_zp55 = pd.read_excel(
    path_zp55, 
    engine='calamine', 
    header=1,
    usecols=colunas_zp55.keys(), 
    dtype=colunas_zp55
)

df_zp54 = pd.read_excel(
    path_zp54, 
    engine='calamine', 
    header=1,
    usecols=colunas_zp54.keys(), 
    dtype=colunas_zp54
)

df_zp53 = pd.read_excel(
    path_zp53, 
    engine='calamine', 
    header=1,
    usecols=colunas_zp53.keys(), 
    dtype=colunas_zp53
)

df_zp52 = pd.read_excel(
    path_zp52, 
    engine='calamine', 
    header=1,
    usecols=colunas_zp52.keys(), 
    dtype=colunas_zp52
)

df_zp73 = pd.read_excel(
    path_zp73, 
    engine='calamine', 
    header=1,
    usecols=colunas_zp73.keys(), 
    dtype=colunas_zp73
)

df_zp70 = pd.read_excel(
    path_zp70, 
    engine='calamine', 
    header=0,
    usecols=colunas_zp70.keys(), 
    dtype=colunas_zp70
)

df_zp39 = pd.read_excel(
    path_zp39, 
    engine='calamine', 
    header=1,
    usecols=colunas_zp39.keys(), 
    dtype=colunas_zp39
)


In [ ]:
df_n13p.columns.to_list()

## Calculos de Valoração do SRM

### 5. Cálculo da ZP55 (Impostos / Alíquota Base)

Nesta etapa, calculamos a **ZP55** aplicando regras de mapeamento hierárquico (Mapping Rules). 

#### Ordem de Prioridade (Cascateamento):
1.  **CLIENTE**: Busca exata por `Company Code` + `COD_CLIENTE` + `Hierarquia`.
2.  **CLIENTE_H10**: Busca por `Company Code` + `COD_CLIENTE` + `Hierarquia` truncada em 10 caracteres.
3.  **CD + UF DESTINO + Importação**: Busca por `CD` + `UF` + `Origem`.
4.  **CD + UF DESTINO + NCM**: Busca por `CD` + `UF` + `NCM`.
5.  **CD + UF DESTINO + H05**: Busca por `CD` + `UF` + `Hierarquia` truncada em 10 caracteres.

Se nenhuma chave encontrar correspondência, o valor padrão adotado é `0`.


In [ ]:
inicio_zp55 = time.time()

# data cleaning
df_zp55_limpo = df_zp55.drop_duplicates(subset=['CHAVE'], keep='first').copy()
df_zp55_limpo['CHAVE'] = df_zp55_limpo['CHAVE'].astype(str).str.strip()

# dic creation
dic_zp55 = df_zp55_limpo.set_index('CHAVE')['Cadastro'].to_dict()


# key construction
company_code = df_n13p['Company Code'].astype(str).str.strip()
cod_cliente = df_n13p['COD_CLIENTE'].astype(str).str.strip()
hierarquia = df_n13p['Hierarquia'].astype(str).str.strip()
cd = df_n13p['CD'].astype(str).str.strip()
uf = df_n13p['UF DESTINO'].astype(str).str.strip()
origem = df_n13p['Origem'].astype(str).str.strip()
ncm = df_n13p['NCM'].astype(str).str.strip()

chave_1 = company_code + '_' + cod_cliente + '_' + hierarquia                  # CLIENTE
chave_2 = company_code + '_' + cod_cliente + '_' + hierarquia.str[:10]         # CLIENTE H05
chave_3 = cd + '_' + uf + '_' + origem                                         # CD + UF + Importação
chave_4 = cd + '_' + uf + '_' + ncm                                            # CD + UF + NCM
chave_5 = cd + '_' + uf + '_' + hierarquia.str[:10]                            # CD + UF + H05


# searching and applying priorities
df_n13p['ZP55 CLIENTE'] = chave_1.map(dic_zp55) / 100
df_n13p['ZP55 CLIENTE H05'] = chave_2.map(dic_zp55) / 100
df_n13p['ZP55 CD + UF DESTINO + Importação'] = chave_3.map(dic_zp55) / 100
df_n13p['ZP55 CD + UF DESTINO + NCM'] = chave_4.map(dic_zp55) / 100
df_n13p['ZP55 CD + UF DESTINO + H05'] = chave_5.map(dic_zp55) / 100

df_n13p['ZP55'] = (df_n13p['ZP55 CLIENTE']
                   .fillna(df_n13p['ZP55 CD + UF DESTINO + Importação'])
                   .fillna(df_n13p['ZP55 CD + UF DESTINO + NCM'])
                   .fillna(df_n13p['ZP55 CD + UF DESTINO + H05'])
                   .fillna(0)  # caso nenhuma das chaves seja encontrada
                  )

colunas_auditoria_zp55 = ['ZP55 CLIENTE', 'ZP55 CLIENTE H05', 'ZP55 CD + UF DESTINO + Importação', 'ZP55 CD + UF DESTINO + NCM', 'ZP55 CD + UF DESTINO + H05']
df_n13p[colunas_auditoria_zp55] = df_n13p[colunas_auditoria_zp55].fillna(0)


fim_zp55 = time.time()
tempo_zp55 = fim_zp55 - inicio_zp55

print(f"⏱️ Tempo de processamento da ZP55: {tempo_zp55:.4f} segundos")
print("✅ Coluna 'ZP55' calculada com sucesso!")
display(df_n13p[['COD_CLIENTE', 'SKU', 'Hierarquia', 'ZP55', 'ZP55 CLIENTE', 'ZP55 CLIENTE H05', 'ZP55 CD + UF DESTINO + Importação', 'ZP55 CD + UF DESTINO + NCM', 'ZP55 CD + UF DESTINO + H05']].head())


### 6. Cálculo da ZP54 (Políticas de Desconto)

Calculamos a **ZP54** aplicando as regras de mapeamento hierárquico com foco no canal de vendas e grupo de clientes (GP).

#### Ordem de Prioridade (Cascateamento):
1.  **1. CLIENTE**: Busca por `Company Code` + `COD_CLIENTE` + `Hierarquia`.
2.  **1. REDE**: Busca por `Company Code` + `COD SUBREDE` + `Hierarquia`.
3.  **1. GP UF HIER 6**: Busca por `Company Code` + `CÓD GP` (separado por espaço da `UF`) + `Hierarquia`.
4.  **1. GP UF HIER 5**: Busca por `Company Code` + `CÓD GP` (separado por espaço da `UF`) + `Hierarquia` truncada em 10 caracteres.


In [ ]:
df_n13p.columns

In [ ]:
inicio_zp54 = time.time()

# data cleaning
df_zp54_limpo = df_zp54.drop_duplicates(subset=['CHAVE'], keep='first').copy()
df_zp54_limpo['CHAVE'] = df_zp54_limpo['CHAVE'].astype(str).str.strip()

# dic creation
dic_zp54 = df_zp54_limpo.set_index('CHAVE')['Cadastro'].to_dict()


# key construction
company_code = df_n13p['Company Code'].astype(str).str.strip()
cod_cliente = df_n13p['COD_CLIENTE'].astype(str).str.strip()
cod_subrede = df_n13p['COD SUBREDE'].astype(str).str.strip()
cod_gp = df_n13p['COD GP'].astype(str).str.strip()
uf = df_n13p['UF DESTINO'].astype(str).str.strip()
hierarquia = df_n13p['Hierarquia'].astype(str).str.strip()


chave_1 = company_code + '_' + cod_cliente + '_' + hierarquia                       # CLIENTE
chave_2 = company_code + '_' + cod_subrede + '_' + hierarquia                       # REDE
chave_3 = company_code + '_' + cod_gp + ' ' + uf + '_' + hierarquia                 # GP UF HIER 6
chave_4 = company_code + '_' + cod_gp + ' ' + uf + '_' + hierarquia.str[:10]        # GP UF HIER 5


# searching and applying priorities
df_n13p['ZP54 CLIENTE'] = chave_1.map(dic_zp54) / 100
df_n13p['ZP54 REDE'] = chave_2.map(dic_zp54) / 100
df_n13p['ZP54 GP UF HIER 6'] = chave_3.map(dic_zp54) / 100
df_n13p['ZP54 GP UF HIER 5'] = chave_4.map(dic_zp54) / 100

df_n13p['ZP54'] = (df_n13p['ZP54 CLIENTE']
                   .fillna(df_n13p['ZP54 REDE'])
                   .fillna(df_n13p['ZP54 GP UF HIER 6'])
                   .fillna(df_n13p['ZP54 GP UF HIER 5'])
                   .fillna(0) # Caso não encontre nenhuma regra
                  )

colunas_auditoria_zp54 = ['ZP54 CLIENTE', 'ZP54 REDE', 'ZP54 GP UF HIER 6', 'ZP54 GP UF HIER 5']
df_n13p[colunas_auditoria_zp54] = df_n13p[colunas_auditoria_zp54].fillna(0)

fim_zp54 = time.time()
tempo_zp54 = fim_zp54 - inicio_zp54

print(f"⏱️ Tempo de processamento da ZP54: {tempo_zp54:.4f} segundos")
print("✅ Coluna 'ZP54' calculada com sucesso!")
display(df_n13p[['COD_CLIENTE', 'SKU', 'Hierarquia', 'ZP54', 'ZP54 CLIENTE', 'ZP54 REDE', 'ZP54 GP UF HIER 6', 'ZP54 GP UF HIER 5']].head())


### 7. Cálculo do GSV (Gross Sales Value)

Nesta etapa, calculamos o faturamento bruto (**GSV**) em duas métricas essenciais para análise de receita e rentabilidade:
1.  **GSV/CDA (Faturamento Bruto por Caixa)**: Aplica sobre o preço de lista (`LSV`) as correções fiscais (`ZP55`) e comerciais (`ZP54`).
2.  **GSV/TON (Faturamento Bruto por Tonelada)**: Converte o valor de caixa para tonelada física, utilizando o peso do SKU (`kg/UN`) e a conversão de caixas (`Unid/CDA`).

*Nota: Utilizamos uma proteção lógica contra divisões por zero ou nulos no cálculo volumétrico.*


In [ ]:
inicio_gsv = time.time()

# GSV = LSV * (1 + ZP55) * (1 + ZP54)
df_n13p['GSV/CDA'] = df_n13p['LSV'] * (1 + df_n13p['ZP55']) * (1 + df_n13p['ZP54'])


# Colunas a serem utilizadas na conversão
coluna_gsv_cda = df_n13p['GSV/CDA'] 
coluna_kg_un = df_n13p['kg/Un'] 
coluna_unid_cda = df_n13p['Unid/CX']

# Calculamos o denominador da conversão física (kg por caixa de despacho)
denominador_peso = coluna_kg_un * coluna_unid_cda

# GSV/TON = (GSV/CDA) / (kg/Un * Unid/CX) * 1000
df_n13p['GSV/TON'] = np.where(
    (denominador_peso == 0) | (denominador_peso.isna()),
    np.nan,                                
    (coluna_gsv_cda / denominador_peso) * 1000           
)

fim_gsv = time.time()
tempo_gsv = fim_gsv - inicio_gsv

print(f"⏱️ Tempo de processamento do GSV: {tempo_gsv:.4f} segundos")
print("✅ Métricas 'GSV/CDA' e 'GSV/TON' calculadas com sucesso!")
display(df_n13p[['COD_CLIENTE', 'SKU', 'LSV', 'ZP55', 'ZP54', 'GSV/CDA', 'GSV/TON']].head())


### 8. Projeções de Receita Bruta (GSV R$ por Período)

Nesta etapa, calculamos o faturamento bruto projetado em reais (**GSV R$**) para cada período fiscal dinâmico do ciclo atual (ex: de `P03-2026` até `P13-2026`).

#### Regra de Projeção:
Para cada período cadastrado, multiplicamos a quantidade projetada de toneladas pela taxa monetária unitária (**GSV/TON**):
$$\text{GSV R\$ } (P_{xx}) = \text{Volume de Demanda } (P_{xx}) \times \text{GSV/TON}$$


In [ ]:
inicio_projecao = time.time()

for p in colunas_periodos:
    
    nome_coluna_projecao = f'GSV R$ {p}'
    
    df_n13p[nome_coluna_projecao] = df_n13p[p] * df_n13p['GSV/TON']

fim_projecao = time.time()
tempo_projecao = fim_projecao - inicio_projecao

colunas_exibicao_projecao = [f'GSV R$ {p}' for p in colunas_periodos[:3]]

print(f"⏱️ Tempo de processamento das projeções: {tempo_projecao:.4f} segundos")
print(f"✅ Projeções dinâmicas calculadas com sucesso para os períodos: {colunas_periodos}")
display(df_n13p[['COD_CLIENTE', 'SKU', 'GSV/TON'] + colunas_periodos[:3] + colunas_exibicao_projecao].head())


### 9. Cálculo da ZP53 (Deduções com Validade de Período)

Calculamos a **ZP53** e sua respectiva data limite de vigência fiscal (**ZP53d**). Esta tabela utiliza uma busca hierárquica por canais e grupos de clientes.

#### Ordem de Prioridade (Cascateamento):
1.  **1. EMISSOR**: Busca por `Company Code` + `COD_CLIENTE` + `Hierarquia`.
2.  **1. REDE**: Busca por `Company Code` + `COD SUBREDE` + `Hierarquia`.
3.  **1. GP UF**: Busca por `Company Code` + `COD GP` (separado por espaço da `UF`) + `Hierarquia`.
4.  **1. GP**: Busca por `Company Code` + `COD GP` + `Hierarquia` truncada em 10 caracteres.


In [ ]:
inicio_zp53 = time.time()

# data cleaning
df_zp53_limpo = df_zp53.drop_duplicates(subset=['CHAVE'], keep='first').copy()
df_zp53_limpo['CHAVE'] = df_zp53_limpo['CHAVE'].astype(str).str.strip()

dic_zp53_cadastro = df_zp53_limpo.set_index('CHAVE')['Cadastro'].to_dict()
dic_zp53_validade = df_zp53_limpo.set_index('CHAVE')["P\'ANO_FIM"].to_dict() # Escapa a aspa simples da coluna


# key construction
company_code = df_n13p['Company Code'].astype(str).str.strip()
cod_cliente = df_n13p['COD_CLIENTE'].astype(str).str.strip()
cod_subrede = df_n13p['COD SUBREDE'].astype(str).str.strip()
cod_gp = df_n13p['COD GP'].astype(str).str.strip() # Ajustado para o nome oficial correto
uf = df_n13p['UF DESTINO'].astype(str).str.strip()
hierarquia = df_n13p['Hierarquia'].astype(str).str.strip()

chave_1 = company_code + '_' + cod_cliente + '_' + hierarquia                       # EMISSOR
chave_2 = company_code + '_' + cod_subrede + '_' + hierarquia                       # REDE
chave_3 = company_code + '_' + cod_gp + ' ' + uf + '_' + hierarquia                 # GP UF
chave_4 = company_code + '_' + cod_gp + '_' + hierarquia.str[:10]                   # GP

# searching and applying priorities
df_n13p['ZP53 EMISSOR'] = chave_1.map(dic_zp53_cadastro) / 100
df_n13p['ZP53 REDE'] = chave_2.map(dic_zp53_cadastro) / 100
df_n13p['ZP53 GP UF'] = chave_3.map(dic_zp53_cadastro) / 100
df_n13p['ZP53 GP'] = chave_4.map(dic_zp53_cadastro) / 100

df_n13p['ZP53'] = (df_n13p['ZP53 EMISSOR']
                   .fillna(df_n13p['ZP53 REDE'])
                   .fillna(df_n13p['ZP53 GP UF'])
                   .fillna(df_n13p['ZP53 GP'])
                   .fillna(0) # Se não encontrar correspondência, adota 0
                  )

df_n13p['ZP53'] = df_n13p['ZP53'].round(4)

df_n13p['ZP53d EMISSOR'] = chave_1.map(dic_zp53_validade)
df_n13p['ZP53d REDE'] = chave_2.map(dic_zp53_validade)
df_n13p['ZP53d GP UF'] = chave_3.map(dic_zp53_validade)
df_n13p['ZP53d GP'] = chave_4.map(dic_zp53_validade)

df_n13p['ZP53d'] = (df_n13p['ZP53d EMISSOR']
                    .fillna(df_n13p['ZP53d REDE'])
                    .fillna(df_n13p['ZP53d GP UF'])
                    .fillna(df_n13p['ZP53d GP'])
                   )


colunas_auditoria_valores = ['ZP53 EMISSOR', 'ZP53 REDE', 'ZP53 GP UF', 'ZP53 GP']
df_n13p[colunas_auditoria_valores] = df_n13p[colunas_auditoria_valores].fillna(0)

fim_zp53 = time.time()
tempo_zp53 = fim_zp53 - inicio_zp53

print(f"⏱️ Tempo de processamento da ZP53: {tempo_zp53:.4f} segundos")
print("✅ Colunas 'ZP53' e 'ZP53d' calculadas com sucesso!")
display(df_n13p[['COD_CLIENTE', 'SKU', 'Hierarquia', 'ZP53', 'ZP53d', 'ZP53 EMISSOR', 'ZP53 REDE', 'ZP53 GP UF', 'ZP53 GP', 'ZP53d EMISSOR', 'ZP53d REDE', 'ZP53d GP UF', 'ZP53d GP']].head(10))


### 10. Cálculo da ZP52 (Deduções Hierárquicas de Produto)

Calculamos a **ZP52** aplicando as regras de mapeamento com base em diferentes níveis de granularidade da hierarquia de produtos (H04 e H01).

#### Ordem de Prioridade (Cascateamento):
1.  **H04**: Busca por `Company Code` + `COD_CLIENTE` + `Hierarquia` truncada em 8 caracteres.
2.  **H01**: Busca por `Company Code` + `COD SUBREDE` + `Hierarquia` truncada em 2 caracteres.


In [ ]:
inicio_zp52 = time.time()


# data cleaning
df_zp52_limpo = df_zp52.drop_duplicates(subset=['CHAVE'], keep='first').copy()
df_zp52_limpo['CHAVE'] = df_zp52_limpo['CHAVE'].astype(str).str.strip()

dic_zp52 = df_zp52_limpo.set_index('CHAVE')['Cadastro'].to_dict()


# key construction
company_code = df_n13p['Company Code'].astype(str).str.strip()
cod_cliente = df_n13p['COD_CLIENTE'].astype(str).str.strip()
cod_subrede = df_n13p['COD SUBREDE'].astype(str).str.strip()
hierarquia = df_n13p['Hierarquia'].astype(str).str.strip()

chave_1 = company_code + '_' + cod_cliente + '_' + hierarquia.str[:8]              # H04
chave_2 = company_code + '_' + cod_subrede + '_' + hierarquia.str[:2]              # H01


# searching and applying priorities
df_n13p['ZP52 H04'] = chave_1.map(dic_zp52) / 100
df_n13p['ZP52 H01'] = chave_2.map(dic_zp52) / 100

df_n13p['ZP52'] = (df_n13p['ZP52 H04']
                   .fillna(df_n13p['ZP52 H01'])
                   .fillna(0) # Adota 0 caso nenhuma regra combine
                  )

df_n13p['ZP52'] = df_n13p['ZP52']


colunas_auditoria_zp52 = ['ZP52 H04', 'ZP52 H01']
df_n13p[colunas_auditoria_zp52] = df_n13p[colunas_auditoria_zp52].fillna(0)

fim_zp52 = time.time()
tempo_zp52 = fim_zp52 - inicio_zp52

print(f"⏱️ Tempo de processamento da ZP52: {tempo_zp52:.4f} segundos")
print("✅ Coluna 'ZP52' calculada com sucesso!")
display(df_n13p[['COD_CLIENTE', 'SKU', 'Hierarquia', 'ZP52', 'ZP52 H04', 'ZP52 H01']].head())


### 11. Cálculo da ZP73 (Acordos Comerciais por Cliente e Rede)

Calculamos a **ZP73** aplicando as regras de precificação estruturadas diretamente para os canais de atendimento e faturamento, sem dependência de agrupamentos de produtos.

#### Ordem de Prioridade (Cascateamento):
1.  **ZP73 CLIENTE**: Busca por `Company Code` + `COD_CLIENTE`.
2.  **ZP73 REDE**: Busca por `Company Code` + `COD SUBREDE`.


In [ ]:
inicio_zp73 = time.time()

# data cleaning
df_zp73_limpo = df_zp73.drop_duplicates(subset=['CHAVE'], keep='first').copy()
df_zp73_limpo['CHAVE'] = df_zp73_limpo['CHAVE'].astype(str).str.strip()

dic_zp73 = df_zp73_limpo.set_index('CHAVE')['Cadastro'].to_dict()


# key construction
company_code = df_n13p['Company Code'].astype(str).str.strip()
cod_cliente = df_n13p['COD_CLIENTE'].astype(str).str.strip()
cod_subrede = df_n13p['COD SUBREDE'].astype(str).str.strip()

chave_1 = company_code + '_' + cod_cliente                                        # CLIENTE
chave_2 = company_code + '_' + cod_subrede                                        # REDE

df_n13p['ZP73 CLIENTE'] = (chave_1.map(dic_zp73) / 100).round(4)
df_n13p['ZP73 REDE'] = (chave_2.map(dic_zp73) / 100).round(4)

df_n13p['ZP73'] = (df_n13p['ZP73 CLIENTE']
                   .fillna(df_n13p['ZP73 REDE'])
                   .fillna(0) # Adota 0 caso nenhuma regra combine
                  )

df_n13p['ZP73'] = df_n13p['ZP73'].round(4)


colunas_auditoria_zp73 = ['ZP73 CLIENTE', 'ZP73 REDE']
df_n13p[colunas_auditoria_zp73] = df_n13p[colunas_auditoria_zp73].fillna(0)

fim_zp73 = time.time()
tempo_zp73 = fim_zp73 - inicio_zp73

print(f"⏱️ Tempo de processamento da ZP73: {tempo_zp73:.4f} segundos")
print("✅ Coluna 'ZP73' calculada com sucesso!")
display(df_n13p[['COD_CLIENTE', 'COD SUBREDE', 'SKU', 'ZP73', 'ZP73 CLIENTE', 'ZP73 REDE']].head(78))


In [ ]:
# Célula de Diagnóstico Corrigida - ZP73

print("===== 📊 AUDITORIA DE VALORES E SOMAS (ZP73) =====")
print(f"Soma total da coluna ZP73 calculada: {df_n13p['ZP73'].sum():.4f}")
print(f"Soma da ZP73 via CLIENTE: {df_n13p['ZP73 CLIENTE'].sum():.4f}")
print(f"Soma da ZP73 via REDE: {df_n13p['ZP73 REDE'].sum():.4f}")

print("\n===== 📈 COBERTURA DE MAPEAMENTO (CORRESPONDÊNCIA REAL) =====")
linhas_totais = len(df_n13p)
# Usando .ne(0) para capturar tanto valores positivos quanto negativos
linhas_com_cliente = df_n13p['ZP73 CLIENTE'].ne(0).sum()
linhas_com_rede = df_n13p['ZP73 REDE'].ne(0).sum()
linhas_com_sucesso_total = df_n13p['ZP73'].ne(0).sum()

print(f"Total de linhas na base mestre: {linhas_totais}")
print(f"Linhas que encontraram regra por CLIENTE: {linhas_com_cliente} ({linhas_com_cliente/linhas_totais*100:.2f}%)")
print(f"Linhas que encontraram regra por REDE: {linhas_com_rede} ({linhas_com_rede/linhas_totais*100:.2f}%)")
print(f"Linhas que encontraram qualquer regra (ZP73 != 0): {linhas_com_sucesso_total} ({linhas_com_sucesso_total/linhas_totais*100:.2f}%)")


In [ ]:
# # Célula de Inspeção - Filtrando ZP73 por valor específico

# # 1. Definimos o valor que queremos inspecionar
# valor_filtro = -0.0735

# # 2. Aplicamos o filtro no DataFrame consolidado
# # Usamos o np.isclose se houver alguma dízima de float, ou comparação direta
# df_filtrado = df_n13p[df_n13p['ZP73'] == valor_filtro]

# print(f"📊 Encontradas {len(df_filtrado)} linhas com o valor ZP73 = {valor_filtro}")

# # 3. Exibimos as colunas que compõem as chaves das linhas filtradas
# # (Limitado às primeiras 10 linhas para não poluir a tela do Jupyter)
# if len(df_filtrado) > 0:
#     colunas_chave = [
#         'Company Code', 'COD_CLIENTE', 'COD SUBREDE', # Componentes das chaves
#         'ZP73 CLIENTE', 'ZP73 REDE', 'ZP73'            # Resultados individuais e final
#     ]
#     display(df_filtrado[colunas_chave].head(10))
# else:
#     print("⚠️ Nenhuma linha encontrada com este valor exato. Tente usar aproximação decimal:")
#     # Filtro por aproximação (caso haja pequenas variações de float na memória)
#     df_filtrado_aprox = df_n13p[np.isclose(df_n13p['ZP73'], valor_filtro, atol=1e-5)]
#     print(f"   Aproximação: {len(df_filtrado_aprox)} linhas encontradas.")
#     if len(df_filtrado_aprox) > 0:
#         display(df_filtrado_aprox[colunas_chave].head(10))


### 12. Cálculo da ZP70 (Descontos por Condição de Pagamento)

Calculamos a **ZP70** mapeando diretamente os prazos e condições financeiras acordadas para cada cliente.

#### Chave de Busca:
*   **COND. PAG**: Busca direta pela condição de pagamento da tabela de clientes contra a base de regras comerciais da ZP70. Caso a condição não exista na tabela de regras, o desconto aplicado é `0`.


In [ ]:
inicio_zp70 = time.time()

# data cleaning
df_zp70_limpo = df_zp70.drop_duplicates(subset=['CONDICAO DE PAGAMENTO'], keep='first').copy()
df_zp70_limpo['CONDICAO DE PAGAMENTO'] = df_zp70_limpo['CONDICAO DE PAGAMENTO'].astype(str).str.strip()


dic_zp70 = df_zp70_limpo.set_index('CONDICAO DE PAGAMENTO')['Desconto'].to_dict()

# key construction
chave_cond_pag = df_n13p['COND. PAG'].astype(str).str.strip()

df_n13p['ZP70'] = chave_cond_pag.map(dic_zp70)
df_n13p['ZP70'] = df_n13p['ZP70'].fillna(0)

fim_zp70 = time.time()
tempo_zp70 = fim_zp70 - inicio_zp70

print(f"⏱️ Tempo de processamento da ZP70: {tempo_zp70:.4f} segundos")
print("✅ Coluna 'ZP70' calculada com sucesso!")
display(df_n13p[['COD_CLIENTE', 'COND. PAG', 'SKU', 'ZP70']].head())

### 13. Cálculo da ZP39 (Descontos Promocionais com Vigência)

Calculamos a **ZP39** (desconto promocional) e a sua data limite de vigência (**ZP39d**).

#### Ordem de Prioridade (Cascateamento):
1.  **39. Emissor H12**: Busca por `Company Code` + `COD_CLIENTE` + `Hierarquia`.
2.  **39. Emissor H10**: Busca por `Company Code` + `COD_CLIENTE` + `Hierarquia` truncada em 10 caracteres.
3.  **39. Subrede H12**: Busca por `Company Code` + `COD SUBREDE` + `Hierarquia`.
4.  **39. GP UF H12**: Busca por `Company Code` + `COD GP` (separado por espaço da `UF`) + `Hierarquia`.


In [ ]:
inicio_zp39 = time.time()

# ==============================================================================
# 1. PREPARAÇÃO DA BASE ZP39 (Dicionários de Busca)
# ==============================================================================
df_zp39_limpo = df_zp39.drop_duplicates(subset=['CHAVE'], keep='first').copy()
df_zp39_limpo['CHAVE'] = df_zp39_limpo['CHAVE'].astype(str).str.strip()

# Criamos dicionários de busca rápidos para o valor de Cadastro e a Vigência
dic_zp39_cadastro = df_zp39_limpo.set_index('CHAVE')['Cadastro'].to_dict()
dic_zp39_validade = df_zp39_limpo.set_index('CHAVE')["P\'ANO_FIM"].to_dict()


# ==============================================================================
# 2. CONSTRUÇÃO DAS CHAVES HIERÁRQUICAS NA BASE MESTRE
# ==============================================================================
company_code = df_n13p['Company Code'].astype(str).str.strip()
cod_cliente = df_n13p['COD_CLIENTE'].astype(str).str.strip()
cod_subrede = df_n13p['COD SUBREDE'].astype(str).str.strip()
cod_gp = df_n13p['COD GP'].astype(str).str.strip() # Ajustado para o nome oficial correto
uf = df_n13p['UF DESTINO'].astype(str).str.strip()
hierarquia = df_n13p['Hierarquia'].astype(str).str.strip()

# Criando as chaves conforme suas MappingRules
chave_1 = company_code + '_' + cod_cliente + '_' + hierarquia                       # Emissor H12
chave_2 = company_code + '_' + cod_cliente + '_' + hierarquia.str[:10]              # Emissor H10
chave_3 = company_code + '_' + cod_subrede + '_' + hierarquia                       # Subrede H12
chave_4 = company_code + '_' + cod_gp + ' ' + uf + '_' + hierarquia                 # GP UF H12


# ==============================================================================
# 3. BUSCA E APLICAÇÃO DE PRIORIDADES
# ==============================================================================
# --- Busca 1: Valores de Cadastro (Dividido por 100 para taxas percentuais) ---
df_n13p['ZP39 Emissor H12'] = (chave_1.map(dic_zp39_cadastro) / 100)
df_n13p['ZP39 Emissor H10'] = (chave_2.map(dic_zp39_cadastro) / 100)
df_n13p['ZP39 Subrede H12'] = (chave_3.map(dic_zp39_cadastro) / 100)
df_n13p['ZP39 GP UF H12'] = (chave_4.map(dic_zp39_cadastro) / 100)

# Executamos o cascateamento para encontrar o valor de desconto final (1. ZP39)
df_n13p['ZP39'] = (df_n13p['ZP39 Emissor H12']
                      .fillna(df_n13p['ZP39 Emissor H10'])
                      .fillna(df_n13p['ZP39 Subrede H12'])
                      .fillna(df_n13p['ZP39 GP UF H12'])
                      .fillna(0) # Se não encontrar, assume 0
                     )

df_n13p['ZP39'] = df_n13p['ZP39'].round(4)

# --- Busca 2: Vigências (2. ZP39) ---
df_n13p['ZP39d Emissor H12'] = chave_1.map(dic_zp39_validade)
df_n13p['ZP39d Emissor H10'] = chave_2.map(dic_zp39_validade)
df_n13p['ZP39d Subrede H12'] = chave_3.map(dic_zp39_validade)
df_n13p['ZP39d GP UF H12'] = chave_4.map(dic_zp39_validade)

# Executamos o cascateamento para encontrar a validade final (2. ZP39)
df_n13p['ZP39d'] = (df_n13p['ZP39d Emissor H12']
                      .fillna(df_n13p['ZP39d Emissor H10'])
                      .fillna(df_n13p['ZP39d Subrede H12'])
                      .fillna(df_n13p['ZP39d GP UF H12'])
                      .fillna('NaN')
                     )


# ==============================================================================
# 4. LIMPEZA E FORMATAÇÃO DE EXIBIÇÃO
# ==============================================================================
# Preenchemos NaNs com zero nas colunas de auditoria de valores apenas para exibição
colunas_auditoria_zp39 = ['ZP39 Emissor H12', 'ZP39 Emissor H10', 'ZP39 Subrede H12', 'ZP39 GP UF H12', 'ZP39', 'ZP39d']
df_n13p[colunas_auditoria_zp39] = df_n13p[colunas_auditoria_zp39].fillna(0)

fim_zp39 = time.time()
tempo_zp39 = fim_zp39 - inicio_zp39

print(f"⏱️ Tempo de processamento da ZP39: {tempo_zp39:.4f} segundos")
print("✅ Colunas '1. ZP39' e '2. ZP39' calculadas com sucesso!")
# display(df_n13p[['COD_CLIENTE', 'SKU', '1. ZP39', '2. ZP39', '39. Emissor H12', '39. Emissor H10']].head())


### 14. Cálculo do NIV (Net Invoice Value / Faturamento Líquido)

Nesta etapa, calculamos a nossa receita líquida final de faturamento (**NIV**), deduzindo do GSV todos os acordos comerciais, descontos logísticos, financeiros e promocionais mapeados pelas ZPs.

#### Fórmulas Aplicadas:
$$\text{NIV/CDA} = \text{GSV/CDA} \times (1 + ZP53) \times (1 + ZP52) \times (1 + ZP73) \times (1 + ZP70) \times (1 + 1. ZP39)$$

$$\text{NIV/TON} = \frac{\text{NIV/CDA}}{\text{kg/UN} \times \text{Unid/CX}} \times 1000$$


In [ ]:
# Célula 14 - Processamento de NIV
inicio_niv = time.time()

# ==============================================================================
# 1. CÁLCULO DO NIV POR CAIXA (NIV/CDA)
# ==============================================================================
# Multiplicação em cascata de todas as ZPs sobre o GSV
df_n13p['NIV/CDA'] = (
    df_n13p['GSV/CDA'] * 
    (1 + df_n13p['ZP53']) * 
    (1 + df_n13p['ZP52']) * 
    (1 + df_n13p['ZP73']) * 
    (1 + df_n13p['ZP70']) * 
    (1 + df_n13p['ZP39'])
)

# Arredondamento estrito em 4 casas decimais reais
df_n13p['NIV/CDA'] = df_n13p['NIV/CDA'].round(4)


# ==============================================================================
# 2. CÁLCULO DO NIV POR TONELADA (NIV/TON)
# ==============================================================================
coluna_niv_cda = df_n13p['NIV/CDA']
coluna_kg_un = df_n13p['kg/Un']       # Usando o nome exato da sua coluna: 'kg/Un'
coluna_unid_cda = df_n13p['Unid/CX']  # Usando o nome exato da sua coluna: 'Unid/CX'

# Calculamos o denominador de conversão física (kg por caixa de despacho)
denominador_peso = coluna_kg_un * coluna_unid_cda

# Divisão segura usando numpy.where
df_n13p['NIV/TON'] = np.where(
    (denominador_peso == 0) | (denominador_peso.isna()),
    np.nan,                                
    (coluna_niv_cda / denominador_peso) * 1000           
)

# Arredondamento estrito em 4 casas decimais reais
df_n13p['NIV/TON'] = df_n13p['NIV/TON'].round(4)

fim_niv = time.time()
tempo_niv = fim_niv - inicio_niv

print(f"⏱️ Tempo de processamento do NIV: {tempo_niv:.4f} segundos")
print("✅ Métricas 'NIV/CDA' e 'NIV/TON' calculadas com sucesso!")
display(df_n13p[['COD_CLIENTE', 'SKU', 'GSV/CDA', 'NIV/CDA', 'NIV/TON']].head())


## Exportação de Teste

In [ ]:
# # Célula de Teste - Exportação

# # 1. Definição do nome do arquivo de saída
# arquivo_saida_teste = caminho_dados / "teste_consolidacao_spim.xlsx"

# # 2. Exportando o DataFrame consolidado para Excel
# # 'index=False' evita que o Pandas salve aquela coluna de índices numéricos (0, 1, 2...) no Excel
# start_export = time.time()
# df_n13p.to_excel(arquivo_saida_teste, index=False)
# end_export = time.time()

# print(f"⏱️ Tempo de exportação: {end_export - start_export:.4f} segundos")
# print(f"💾 Arquivo salvo com sucesso em: {arquivo_saida_teste.resolve()}")
# print(f"📊 Dimensões atuais da base: {df_n13p.shape[0]} linhas e {df_n13p.shape[1]} colunas.")


In [ ]:
df_n13p.columns.tolist()

## 15. Exportação Estruturada Final (Marco de Valoração)

Nesta etapa, consolidamos o nosso DataFrame mestre ordenando todas as colunas de forma lógica e intuitiva para o usuário de negócio no Excel.

### Estrutura de Fluxo do Layout:
1.  **Dados Cadastrais do Ciclo**: Colunas originais da N13P (Tipo, Clientes, Geografia, SKU).
2.  **Dados Físicos e Comerciais de Suporte**: Dados consolidados de pesos, embalagem e tabela (LSV).
3.  **Auditoria e Resultado ZP55**: Alíquotas básicas de impostos.
4.  **Auditoria e Resultado ZP54**: Regras de políticas comerciais de desconto.
5.  **Valoração Bruta (GSV)**: Preço de faturamento bruto (GSV/CDA e GSV/TON).
6.  **Projeções Dinâmicas de Demanda**: Faturamento projetado em reais (GSV R$) para cada período ativo.
7.  **Deduções Comerciais (ZP53, ZP52, ZP73, ZP70, ZP39)**: Toda a cascata de validação de descontos.
8.  **Valoração Líquida Final (NIV)**: Faturamento líquido definitivo (NIV/CDA e NIV/TON).

### 16. Formatação Visual de Estilos e Cores (XlsxWriter)

Nesta etapa final do marco de exportação, aplicamos uma identidade visual profissional ao arquivo gerado utilizando a biblioteca **XlsxWriter**. 

### Mapeamento de Cores dos Cabeçalhos:
*   ⚪ **Branco (Padrão)**: Dados cadastrais básicos da planilha Ciclo N13P.
*   💜 **Roxo**: Cadastro técnico de produtos e clientes.
*   🔴 **Vermelho**: A chave inteligente de barramento `EAN Espelho`.
*   🔵 **Azul**: As colunas de resultados finais das tabelas ZPs.
*   🐳 **Ciano (Água)**: Métricas de faturamento principais (GSV e NIV).
*   🟢 **Verde**: Colunas de projeções de demanda monetárias (`GSV R$`).
*   ⚫ **Cinza**: Colunas de auditoria física e lógica das chaves (ZPs de cascateamento).

### Formatação de Dados:
*   **Precisão Decimal**: Força as células numéricas das colunas de cadastro e ZPs a exibirem fisicamente 4 casas decimais no Excel (`0.0000`).
*   **Largura Auto-Ajustável**: Redimensiona a largura das colunas dinamicamente baseado no tamanho dos textos para evitar textos cortados (`###`).



In [ ]:
ordem_desejada_colunas = [
    # Colunas do Ciclo N13P
    'Tipo 1', 'Tipo 2', 'Tipo 3', 'Regional', 'GP', 'Vend.', 'Gerente', 'Rede', 'COD_CLIENTE',
    'Company Code', 'CD', 'NOME_CLIENTE', 'UF DESTINO', 'Região', 'EAN', 'SKU', 'Desc. SKU', 'Classificação',
    'Tech', 'Tech 2', 'Subbrand', 'Size', 'Nivel 3 HieraR', 'Marca',

    # Colunas de Produtos e Clientes
    'kg/Un',  'COND. PAG', 'COD REDE', 'COD SUBREDE', 'UF ORIGEM', 'COD GP', 'EAN Espelho', 'Family Price', 'Hierarquia', 'Class.', 'NCM', 'Origem', 
    'Ton/CDA', 'Unid/CX', 'LSV',

    # ZP55 
    'ZP55', 'ZP55 CLIENTE', 'ZP55 CLIENTE H05', 'ZP55 CD + UF DESTINO + Importação', 'ZP55 CD + UF DESTINO + NCM', 'ZP55 CD + UF DESTINO + H05',

    # ZP54 
    'ZP54', 'ZP54 CLIENTE', 'ZP54 REDE', 'ZP54 GP UF HIER 6', 'ZP54 GP UF HIER 5', 

    # GSV
    'GSV/CDA', 'GSV/TON',
    
    # ZP53
    'ZP53', 'ZP53 EMISSOR', 'ZP53 REDE', 'ZP53 GP UF', 'ZP53 GP', 
    'ZP53d','ZP53d EMISSOR', 'ZP53d REDE', 'ZP53d GP UF', 'ZP53d GP',

    # ZP52 
    'ZP52', 'ZP52 H04', 'ZP52 H01',

    # ZP73 e ZP70
    'ZP73', 'ZP73 CLIENTE', 'ZP73 REDE', 'ZP70',

    # ZP39
    'ZP39', 'ZP39 Emissor H12', 'ZP39 Emissor H10', 'ZP39 Subrede H12', 'ZP39 GP UF H12', 
    'ZP39d','ZP39d Emissor H12', 'ZP39d Emissor H10', 'ZP39d Subrede H12', 'ZP39d GP UF H12',

    # NIV
    'NIV/CDA', 'NIV/TON'
]

df_n13p_exportar = df_n13p.copy()

colunas_presentes = [col for col in ordem_desejada_colunas if col in df_n13p_exportar.columns]
colunas_faltantes = [col for col in df_n13p_exportar.columns if col not in colunas_presentes]

ordem_final_colunas = colunas_presentes + colunas_faltantes
df_n13p_ordenado = df_n13p_exportar[ordem_final_colunas]

In [ ]:
df_n13p_ordenado.head(10)

In [ ]:
inicio_export = time.time()

# Categorias de colunas para estilização
col_branco = [
    'Tipo 1', 'Tipo 2', 'Tipo 3', 'Regional', 'GP', 'Vend.', 'Gerente', 'Rede', 'COD_CLIENTE',
    'Company Code', 'CD', 'NOME_CLIENTE', 'UF DESTINO', 'Região', 'EAN', 'SKU', 'Desc. SKU', 'Classificação',
    'Tech', 'Tech 2', 'Subbrand', 'Size', 'Nivel 3 HieraR', 'Marca'
]

col_roxo = [
    'COND. PAG', 'COD REDE', 'COD SUBREDE', 'COD GP', 'Family Price', 'Hierarquia', 
    'Class.', 'NCM', 'Origem', 'kg/Un', 'Ton/CDA', 'Unid/CX', 'LSV', 'UF ORIGEM'
]

col_vermelho = 'EAN Espelho'

# Resultados Consolidados das ZPs (Azul)
col_azul = ['ZP55', 'ZP54', 'ZP53', 'ZP53d', 'ZP52', 'ZP73', 'ZP70', 'ZP39', 'ZP39d']

# Indicadores Financeiros de Faturamento (Ciano/Água)
col_agua = ['GSV/CDA', 'GSV/TON', 'NIV/CDA', 'NIV/TON']

# Colunas de Auditoria e Chaves de Validação (Cinza)
col_cinza = [
    # ZP55
    'ZP55 CLIENTE', 'ZP55 CLIENTE H05', 'ZP55 CD + UF DESTINO + Importação', 'ZP55 CD + UF DESTINO + NCM', 'ZP55 CD + UF DESTINO + H05',
    # ZP54
    'ZP54 CLIENTE', 'ZP54 REDE', 'ZP54 GP UF HIER 6', 'ZP54 GP UF HIER 5',
]

col_c_escuro = [
    # ZP53
    'ZP53 EMISSOR', 'ZP53 REDE', 'ZP53 GP UF', 'ZP53 GP', 'ZP53d EMISSOR', 'ZP53d REDE', 'ZP53d GP UF', 'ZP53d GP',
    # ZP52
    'ZP52 H04', 'ZP52 H01',
    # ZP73
    'ZP73 CLIENTE', 'ZP73 REDE',
    # ZP39
    'ZP39 Emissor H12', 'ZP39 Emissor H10', 'ZP39 Subrede H12', 'ZP39 GP UF H12',
    'ZP39d Emissor H12', 'ZP39d Emissor H10', 'ZP39d Subrede H12', 'ZP39d GP UF H12']

# Colunas que necessitam de formatação numérica
col_int = ['Unid/CX',]
col_2d = ['kg/Un', 'LSV',]
col_2d = col_2d + col_cinza + col_c_escuro + col_azul
col_8d = col_agua

# nome do arquivo

nome_arquivo_formatado = 'N13_Final_7.0.xlsx'

###### Exportação  em chunks######
inicio_export = time.time()

writer = pd.ExcelWriter(nome_arquivo_formatado, engine='xlsxwriter')

total_linhas = len(df_n13p)

print(f"total de linhas a serem salvas: {total_linhas}")

tamanho_chunk = 35000
progresso_inicial = 0
progresso_final = 100
margem_progresso = progresso_final - progresso_inicial

for i in range(0, total_linhas, tamanho_chunk):
                # Corta o dataframe no pedaço atual
                chunk = df_n13p.iloc[i : i + tamanho_chunk]
                # Se for o primeiro pedaço, grava com cabeçalho.
                # Se forem os próximos, começamos na linha abaixo (i + 1) e pulamos o cabeçalho (header=False)
                if i == 0:
                    chunk.to_excel(writer, sheet_name='Valoracao', index=False, startrow=0, header=True)
                else:
                    chunk.to_excel(writer, sheet_name='Valoracao', index=False, startrow=i + 1, header=False)
                # Calcula o progresso dinâmico das linhas salvas
                linhas_processadas = min(i + tamanho_chunk, total_linhas)
                porcentagem_linhas = linhas_processadas / total_linhas
                # Transforma isso na escala de 50% a 90% da barra
                progresso_atual = int(progresso_inicial + (porcentagem_linhas * margem_progresso))
                # Atualiza a tela do usuário!
                print(f"Salvando linhas no Excel: {linhas_processadas:,} de {total_linhas:,} concluídas... Total: {progresso_atual}%")

##### Personalização do estilo do Excel #####

workbook  = writer.book
worksheet = writer.sheets['Valoracao']

estilo_header = {
    'bold': True, 'top': 1, 'top_color': '#000000',
    # 'bottom': 1, 'bottom_color': '#000000', 
    'align': 'right', 'valign': 'vcenter', 
    'font_name': 'Mars Centra'}

fmt_branco   = workbook.add_format({**estilo_header, 'bg_color': '#FFFFFF', 'font_color': '#000000'})
fmt_roxo     = workbook.add_format({**estilo_header, 'bg_color': "#A02B93", 'font_color': "#FFFFFF"})
fmt_vermelho = workbook.add_format({**estilo_header, 'bg_color': "#FF0000", 'font_color': "#FFFFFF"})
fmt_azul     = workbook.add_format({**estilo_header, 'bg_color': "#0070C0", 'font_color': "#FFFFFF"})
fmt_cinza    = workbook.add_format({**estilo_header, 'bg_color': "#D9D9D9", 'font_color': "#000000"})
fmt_c_escuro = workbook.add_format({**estilo_header, 'bg_color': "#808080", 'font_color': "#FFFFFF"})
fmt_agua     = workbook.add_format({**estilo_header, 'bg_color': "#CAEDFB", 'font_color': "#000000"})
fmt_verde    = workbook.add_format({**estilo_header, 'bg_color': "#06E92C", 'font_color': "#000000"})

col_n13p_fim = 'Marca'
fmt_header_n13p_fim    = workbook.add_format({**estilo_header, 'bg_color': '#FFFFFF', 'font_color': '#000000', 'right': 1, 'right_color': '#000000'})
fmt_n13p_fim           = workbook.add_format({'right': 1, 'right_color': '#000000'})

# Formatos de células de dados (Corpo da Planilha)
fmt_texto_padrao   = workbook.add_format({'font_name': 'Mars Centra'})
fmt_integer_2 = workbook.add_format({'font_name': 'Aptos Narrow', 'num_format': '0'})
fmt_decimal_2 = workbook.add_format({'font_name': 'Aptos Narrow', 'num_format': '0.00'})
fmt_decimal_8 = workbook.add_format({'font_name': 'Aptos Narrow', 'num_format': '0.00000000'})

for col_num, col_nome in enumerate(df_n13p_ordenado.columns):
    # nome da coluna
    if col_nome in col_branco:
        formato = fmt_branco
    elif col_nome in col_roxo:
        formato = fmt_roxo
    elif col_nome == col_vermelho:
        formato = fmt_vermelho
    elif col_nome in col_azul:
        formato = fmt_azul
    elif col_nome in col_agua:
        formato = fmt_agua
    elif col_nome in col_cinza:
        formato = fmt_cinza
    elif col_nome in col_c_escuro:
        formato = fmt_c_escuro
    elif col_nome.startswith('GSV R$'):
        formato = fmt_branco
    elif col_nome == col_n13p_fim:
        formato = fmt_header_n13p_fim
    else:
        formato = fmt_branco

    worksheet.write(0, col_num, col_nome, formato)

    # column width

    max_comprimento_dados = df_n13p_ordenado[col_nome].head(30000).astype(str).str.len().max()
    largura = max(len(col_nome), max_comprimento_dados) + 3
    largura = min(largura, 50)
   

    # float formatting

    if col_nome in col_2d and df_n13p_ordenado[col_nome].dtype in [np.float64, np.float32]:
        worksheet.set_column(col_num, col_num, largura, fmt_decimal_2)
    elif col_nome in col_8d and df_n13p_ordenado[col_nome].dtype in [np.float64, np.float32]:
        worksheet.set_column(col_num, col_num, largura, fmt_decimal_8)
    elif col_nome in col_int and df_n13p_ordenado[col_nome].dtype in [np.int64, np.int32]:
        worksheet.set_column(col_num, col_num, largura, fmt_integer_2)
    elif col_nome == col_n13p_fim:
        worksheet.set_column(col_num, col_num, largura, fmt_n13p_fim)
    else:
        worksheet.set_column(col_num, col_num, largura, fmt_texto_padrao)

# save
writer.close()
fim_export = time.time()
tempo_exportacao = fim_export - inicio_export
print("===== 🏆 ARQUIVO ESTILIZADO SALVO COM SUCESSO! =====")
print(f"⏱️ Tempo total de processamento: {tempo_exportacao:.2f} segundos")
print(f"💾 Arquivo salvo como: {Path(nome_arquivo_formatado).resolve()}")